### streamlit 사용을 위한 패키지 설치

In [ ]:
# 1. 패키지 설치
pip install streamlit sentence-transformers chromadb faiss-cpu langchain langchain-community langchain-huggingface datasets

# 2. 앱 실행
streamlit run app.py

#### app.py 내용

- RAG 챗봇 — allganize/RAG-Evaluation-Dataset-KO (medical 60건)
- ChromaDB + ko-sroberta 임베딩 + st.chat_input/st.chat_message


In [ ]:
import streamlit as st
import time
from datasets import load_dataset
import chromadb
from chromadb.utils import embedding_functions

# ── 페이지 설정 ──────────────────────────────────────────
st.set_page_config(
    page_title="의료 RAG 챗봇",
    page_icon="🏥",
    layout="centered",
)

st.title("🏥 의료 문서 RAG 챗봇")
st.caption("allganize/RAG-Evaluation-Dataset-KO · medical 60건 · ko-sroberta 임베딩 · ChromaDB")

# ── 데이터 + 인덱스 초기화 (한 번만) ────────────────────
@st.cache_resource(show_spinner="데이터 로드 및 ChromaDB 인덱싱 중...")
def build_index():
    # 1. 데이터 로드
    ds = load_dataset("allganize/RAG-Evaluation-Dataset-KO")
    df = ds["test"].to_pandas()

    # 2. medical 필터링
    df_medical = df[df["domain"] == "medical"].copy()
    df_medical["ids"] = ["id" + str(i + 1) for i in df_medical.index]
    reports = df_medical.to_dict(orient="records")

    # 3. ChromaDB 인덱싱
    ko_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="jhgan/ko-sroberta-multitask"
    )
    client = chromadb.Client()
    collection = client.get_or_create_collection(
        name="medical_reports",
        embedding_function=ko_ef,
    )

    documents = [r["target_answer"] for r in reports]
    ids       = [r["ids"] for r in reports]
    collection.add(documents=documents, ids=ids)

    return collection, reports

collection, reports = build_index()

# ── 사이드바 ─────────────────────────────────────────────
with st.sidebar:
    st.markdown("###설정")
    k = st.slider("검색 문서 수 (k)", 1, 5, 3)
    st.markdown("---")
    st.markdown("**임베딩 모델**")
    st.code("jhgan/ko-sroberta-multitask")
    st.markdown("**벡터스토어**")
    st.code("ChromaDB (in-memory)")
    st.markdown("**데이터**")
    st.code(f"medical {len(reports)}건")
    st.markdown("---")
    if st.button("대화 초기화"):
        st.session_state.messages = []
        st.rerun()

# ── 세션 초기화 ───────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = []

# ── 대화 히스토리 표시 ────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if msg["role"] == "assistant" and "sources" in msg:
            with st.expander(f"참고 문서 {len(msg['sources'])}건 (거리 기준)"):
                for src in msg["sources"]:
                    st.markdown(f"**`{src['id']}`** · 거리: `{src['dist']:.4f}`")
                    st.info(src["text"])

# ── 입력창 ────────────────────────────────────────────────
if prompt := st.chat_input("의료 관련 질문을 입력하세요..."):

    # 사용자 메시지 표시
    with st.chat_message("user"):
        st.markdown(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})

    # 검색 + 응답
    with st.chat_message("assistant"):
        with st.spinner("검색 중..."):
            start = time.time()
            results = collection.query(query_texts=[prompt], n_results=k)
            elapsed = (time.time() - start) * 1000

        # 검색 결과 조합
        docs  = results["documents"][0]
        dists = results["distances"][0]
        ids_  = results["ids"][0]

        # 컨텍스트 → 답변 구성 (목업: 검색 결과 그대로 표시)
        context = "\n\n".join(
            f"[{i+1}] {doc}" for i, doc in enumerate(docs)
        )
        answer = f"검색된 문서를 바탕으로 답변드립니다.\n\n{context}"

        st.markdown(answer)
        st.caption(f"검색 시간: {elapsed:.1f}ms · k={k}")

        sources = [
            {"id": id_, "dist": dist, "text": doc}
            for id_, dist, doc in zip(ids_, dists, docs)
        ]
        with st.expander(f"참고 문서 {len(sources)}건 (거리 기준)"):
            for src in sources:
                st.markdown(f"**`{src['id']}`** · 거리: `{src['dist']:.4f}`")
                st.info(src["text"])

    # 세션 저장
    st.session_state.messages.append({
        "role": "assistant",
        "content": answer,
        "sources": sources,
    })
